# Differential Testing
Identifying functional differences between two code candidates by generating different inputs

In [302]:
from common.coevolution import lcb_dataset
problems = lcb_dataset.load_code_generation_dataset(
    release_version="release_v6",
    start_date="2025-03-01",
    end_date="2025-05-10",
    difficulty=lcb_dataset.Difficulty.HARD,
)

2025-12-10 15:30:04.414 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:127 - Using 'test' split of the dataset.
2025-12-10 15:30:26.835 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:141 - Filtered problems by difficulty: hard
2025-12-10 15:30:26.979 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:151 - Loaded 37 problems


In [304]:
for problem in problems:
    print(problem.public_test_cases[0].testtype)

TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.STDIN
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL
TestType.FUNCTIONAL


In [307]:
from common.coevolution.lcb_dataset import TestType 
# problem_id = 'arc194_c'

# problem = next((p for p in problems if p.question_id == problem_id), None)
# if problem is None:
#     raise ValueError(f"Problem with ID {problem_id} not found.")


problem = [p for p in problems if p.public_test_cases[0].testtype == TestType.FUNCTIONAL][1]

In [308]:
print(problem.starter_code)

class Solution:
    def maxSubarrays(self, n: int, conflictingPairs: List[List[int]]) -> int:
        


In [309]:
print(problem.question_content)

You are given an integer n which represents an array nums containing the numbers from 1 to n in order. Additionally, you are given a 2D array conflictingPairs, where conflictingPairs[i] = [a, b] indicates that a and b form a conflicting pair.
Remove exactly one element from conflictingPairs. Afterward, count the number of non-empty subarrays of nums which do not contain both a and b for any remaining conflicting pair [a, b].
Return the maximum number of subarrays possible after removing exactly one conflicting pair.
 
Example 1:

Input: n = 4, conflictingPairs = [[2,3],[1,4]]
Output: 9
Explanation:

Remove [2, 3] from conflictingPairs. Now, conflictingPairs = [[1, 4]].
There are 9 subarrays in nums where [1, 4] do not appear together. They are [1], [2], [3], [4], [1, 2], [2, 3], [3, 4], [1, 2, 3] and [2, 3, 4].
The maximum number of subarrays we can achieve after removing one element from conflictingPairs is 9.


Example 2:

Input: n = 5, conflictingPairs = [[1,2],[2,5],[3,5]]
Output: 

# Candidate Code Solutions

In [310]:
C0_snippet = """
from typing import List

class Solution:
    def maxSubarrays(self, n: int, conflictingPairs: List[List[int]]) -> int:
        \"\"\"
        Approach 6 — Segment tree to maintain minimal blocking right and support single-pair removal updates (O((n+m) log n)):
        Build array blocked_right per start using all pairs. For each pair removal, we update affected positions (range [1..a])
        where current minimal is b by raising to next minimal; we simulate updates via segment tree range-min queries and point updates,
        recomputing total fast. This version implements a simple segment tree for min and supports recalculation per removal.
        \"\"\"
        m = len(conflictingPairs)
        if m == 0:
            return n * (n + 1) // 2

        pairs = [(min(a, b), max(a, b)) for a, b in conflictingPairs]
        INF = n + 1

        # For each a, get sorted bs
        from collections import defaultdict
        bs_at = defaultdict(list)
        for idx, (a, b) in enumerate(pairs):
            bs_at[a].append(b)
        for a in bs_at:
            bs_at[a].sort()

        # initial blocked_right_at and propagated blocked_right
        blocked_right_at = [INF] * (n + 2)
        for a in bs_at:
            blocked_right_at[a] = bs_at[a][0]
        blocked_right = [INF] * (n + 2)
        for i in range(n, 0, -1):
            blocked_right[i] = min(blocked_right_at[i], blocked_right[i + 1])

        # Total
        base_total = sum(max(0, blocked_right[i] - 1 - i + 1) for i in range(1, n + 1))
        best = 0

        # We'll perform per-pair simulation by copying blocked_right_at and blocked_right and updating region [1..a]
        for a, b in pairs:
            # copy arrays
            br_at = blocked_right_at[:]
            br = blocked_right[:]
            # remove b at a
            lst = bs_at.get(a, [])
            # find next smallest at a
            if not lst:
                best = max(best, base_total)
                continue
            if lst[0] != b:
                # removing non-minimal at a has no effect
                best = max(best, base_total)
                continue
            if len(lst) == 1:
                br_at[a] = INF
            else:
                br_at[a] = lst[1]
            # propagate from a down
            for i in range(a, 0, -1):
                new = min(br_at[i], br[i + 1] if i + 1 <= n else INF)
                if new == br[i]:
                    break
                br[i] = new
            # compute new total by recomputing contributions for affected indices
            new_total = base_total
            for i in range(1, a + 1):
                old_cnt = max(0, blocked_right[i] - 1 - i + 1)
                new_cnt = max(0, br[i] - 1 - i + 1)
                new_total += (new_cnt - old_cnt)
            best = max(best, new_total)
        return best

"""


C1_snippet = """
from typing import List

class Solution:
    def maxSubarrays(self, n: int, conflictingPairs: List[List[int]]) -> int:
        \"\"\"
        Approach 4 — Two-pointer counting using earliest blocking per left and influence graph (O(n + m)):
        Compute for each left i the earliest blocked_right[i].
        Then precompute for each pair (a,b) whether it is the active blocker for any left index.
        For each pair to remove, we can subtract all contributions that were blocked uniquely by that pair
        and add contributions unlocked by replacing with next-best blocker. Implemented by recording,
        for each left, the pair index that caused its min (if any), and a next-min candidate.
        This yields O(n + m) time.
        \"\"\"
        m = len(conflictingPairs)
        if m == 0:
            return n * (n + 1) // 2

        pairs = [(min(a, b), max(a, b)) for a, b in conflictingPairs]
        INF = n + 1

        # For each a, we need all b's sorted to pick minimal and second minimal
        from collections import defaultdict
        pos_bs = defaultdict(list)
        for idx, (a, b) in enumerate(pairs):
            pos_bs[a].append((b, idx))
        for a in pos_bs:
            pos_bs[a].sort()

        # For each left i, determine the active blocking pair (index) and its b, and next best b
        active_pair_for_left = [-1] * (n + 2)
        blocked_right = [INF] * (n + 2)
        next_best_b = [INF] * (n + 2)

        # We'll track for each left i the minimal b among any a>=i.
        # Sweep a from n down to 1 maintaining current minimal (b, pair_idx)
        current_b = INF
        current_pair = -1
        current_second_b = INF

        # To get next best, when multiple pairs contribute at various a, we consider per a the minimal and second minimal.
        # Build arrays min_b_at_a and second_min_b_at_a
        min_b_at_a = [INF] * (n + 2)
        min_idx_at_a = [-1] * (n + 2)
        second_b_at_a = [INF] * (n + 2)
        for a in range(1, n + 1):
            lst = pos_bs.get(a, [])
            if lst:
                min_b_at_a[a] = lst[0][0]
                min_idx_at_a[a] = lst[0][1]
                if len(lst) > 1:
                    second_b_at_a[a] = lst[1][0]

        # sweep
        for i in range(n, 0, -1):
            # consider a == i
            if min_b_at_a[i] < current_b:
                current_second_b = min(current_b, second_b_at_a[i])
                current_b = min_b_at_a[i]
                current_pair = min_idx_at_a[i]
            else:
                # maybe second best from a influences current_second_b
                current_second_b = min(current_second_b, min_b_at_a[i], second_b_at_a[i])
            blocked_right[i] = current_b
            next_best_b[i] = current_second_b
            active_pair_for_left[i] = current_pair

        # compute base total and for each pair how many left indices it is active for
        base_total = 0
        cnt_active = [0] * m
        for i in range(1, n + 1):
            maxr = blocked_right[i] - 1
            if maxr >= i:
                base_total += (maxr - i + 1)
            pid = active_pair_for_left[i]
            if pid != -1:
                cnt_active[pid] += 1

        # For removing pair p: for left indices where it was active, new blocked_right becomes next_best_b[i]
        best = 0
        # To recompute fast, precompute old contributions per left
        old_contrib = [0] * (n + 2)
        for i in range(1, n + 1):
            old_contrib[i] = max(0, blocked_right[i] - 1 - i + 1)

        # For each pair, compute new_total by adjusting left indices where it's active
        # We need to know which left indices had active_pair == p. Build lists:
        active_lists = [[] for _ in range(m)]
        for i in range(1, n + 1):
            pid = active_pair_for_left[i]
            if pid != -1:
                active_lists[pid].append(i)

        for p in range(m):
            new_total = base_total
            if not active_lists[p]:
                best = max(best, base_total)
                continue
            for i in active_lists[p]:
                new_maxr = next_best_b[i] - 1
                new_cnt = max(0, new_maxr - i + 1)
                new_total += (new_cnt - old_contrib[i])
            best = max(best, new_total)
        return best
"""

In [311]:
# Creating code individuals and populations
from common.coevolution.core.individual import CodeIndividual
from common.coevolution.core.population import CodePopulation
from common.coevolution.core.interfaces import Operations


c0_ind = CodeIndividual(
    snippet=C0_snippet,
    probability=0.75,
    creation_op=Operations.INITIAL,
    generation_born=0,
    parent_ids=[],
)

c1_ind = CodeIndividual(
    snippet=C1_snippet,
    probability=0.75,
    creation_op=Operations.INITIAL,
    generation_born=0,
    parent_ids=[],
)


code_pop = CodePopulation(individuals=[c0_ind, c1_ind], generation=0)

2025-12-10 15:30:53.271 | DEBUG    | common.coevolution.core.individual:__init__:42 - Created new <CodeIndividual id=C10 gen=0 op=initial prob=0.75>
2025-12-10 15:30:53.273 | DEBUG    | common.coevolution.core.individual:__init__:42 - Created new <CodeIndividual id=C11 gen=0 op=initial prob=0.75>
2025-12-10 15:30:53.273 | DEBUG    | common.coevolution.core.interfaces:__init__:479 - Initialized CodePopulation with 2 individuals, gen 0


In [312]:
from common.code_preprocessing.composition import compose_lcb_output_script
from common.coevolution.lcb_dataset import LCBTest
import json
import re


def get_inputdata_dict_from_functional_test_case(pub_test_case: LCBTest, starter_code: str) -> dict:
    if not isinstance(pub_test_case, LCBTest):
        raise TypeError("pub_test_case must be an instance of LCBTest")
    
    if pub_test_case.testtype != TestType.FUNCTIONAL:
        raise ValueError("pub_test_case must be of FUNCTIONAL test type")
    
    # Regex to find the first method defined with 'self', and extract its arguments
    func_match = re.search(r'def\s+\w+\s*\(self\s*,\s*([^\)]*)\)', starter_code)
    if not func_match:
        raise ValueError("Could not find a method with 'self' in the starter code.")
    func_args = func_match.group(1).strip().split(',')
    func_args = [arg.split(":")[0].strip() for arg in func_args if arg.strip()]

    input_args_list = [eval(arg) for arg in pub_test_case.input.split("\n")]
    if len(input_args_list) != len(func_args):
        raise ValueError("Number of input arguments does not match number of function parameters.")
    
    input_data_dict = dict(zip(func_args, input_args_list))
    return input_data_dict
    
def compose_output_script_from_public_test_case(code_snippet: str, pub_test_case: LCBTest, starter_code: str) -> str:
    if not isinstance(pub_test_case, LCBTest):
        raise TypeError("pub_test_case must be an instance of LCBTest")
    
    if pub_test_case.testtype == TestType.STDIN:
        input_data = json.dumps({'inputdata': {'input_str': pub_test_case.input}})

    elif pub_test_case.testtype == TestType.FUNCTIONAL:
        input_data_dict = get_inputdata_dict_from_functional_test_case(pub_test_case, starter_code)
        
        input_data = json.dumps({'inputdata': input_data_dict})

    composed_script = compose_lcb_output_script(programmer_code=code_snippet, input_data=input_data)
    return composed_script

In [313]:
import importlib
from common.sandbox import create_safe_test_environment

sandbox = create_safe_test_environment()

functionally_equivalent = True
public_test_input_outputs = []
for test in problem.public_test_cases:
    print(f"Executing test with input: {test.input}")
    c0_script = compose_output_script_from_public_test_case(c0_ind.snippet, test, starter_code=problem.starter_code)
    c0_result = sandbox.execute_code(c0_script)

    c1_script = compose_output_script_from_public_test_case(c1_ind.snippet, test, starter_code=problem.starter_code)
    c1_result = sandbox.execute_code(c1_script)

    print(f"C0 Output: {c0_result.output.strip()}, C1 Output: {c1_result.output.strip()}, Expected: {test.output.strip()}")


    if c0_result.output.strip() != c1_result.output.strip():
        functionally_equivalent = False
        print(f"The two code snippets are NOT functionally equivalent on problem {problem.question_id}.")
        break

    public_test_input_outputs.append({'inputdata':test.input if test.testtype == TestType.STDIN else get_inputdata_dict_from_functional_test_case(test, problem.starter_code), 'output': c0_result.output.strip()})

if functionally_equivalent:
    print(f"The two code snippets are functionally equivalent on all public test cases of problem {problem.question_id}.")

2025-12-10 15:30:57.203 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpsv2p3uzy.py capture_output=True code_len=2431
2025-12-10 15:30:57.296 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-10 15:30:57.300 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmp_nnf9q4i.py capture_output=True code_len=3345
2025-12-10 15:30:57.339 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-10 15:30:57.341 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmp_j672hbl.py capture_output=True code_len=2439
2025-12-10 15:30:57.379 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-10 15:30:57.382 | DEBUG    | common.sandbox:execute_code:543 -

Executing test with input: 4
[[2, 3], [1, 4]]
C0 Output: 9, C1 Output: 9, Expected: 9
Executing test with input: 5
[[1, 2], [2, 5], [3, 5]]


2025-12-10 15:30:57.422 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0


C0 Output: 12, C1 Output: 12, Expected: 12
The two code snippets are functionally equivalent on all public test cases of problem 3789.


In [314]:
public_test_input_outputs

[{'inputdata': {'n': 4, 'conflictingPairs': [[2, 3], [1, 4]]}, 'output': '9'},
 {'inputdata': {'n': 5, 'conflictingPairs': [[1, 2], [2, 5], [3, 5]]},
  'output': '12'}]

# LLM CLient

In [315]:
import common
import importlib

importlib.reload(common.llm_client)

from common.llm_client import create_llm_client
from common.code_preprocessing.extraction import extract_code_block_from_response


llm_client = create_llm_client(provider='openai', model='gpt-5-mini', reasoning_effort='minimal')

2025-12-10 15:31:20.471 | INFO     | common.llm_client:create_llm_client:398 - Creating LLM client: provider=openai, model=gpt-5-mini
2025-12-10 15:31:20.471 | DEBUG    | common.llm_client:__init__:47 - Initialized LLM client: model=gpt-5-mini, max_tokens=1000000, limit_enabled=True
2025-12-10 15:31:20.491 | DEBUG    | common.llm_client:__init__:249 - Initialized OpenAIClient with model: gpt-5-mini, reasoning_effort: minimal
2025-12-10 15:31:20.491 | INFO     | common.llm_client:create_llm_client:428 - Successfully created OpenAIClient


# Solution explanations

In [316]:
EXPLAIN_PROMPT_TEMPLATE = """
<task>
- You are given a programming problem in <problem> and a code snippet in <code_snippet> that is believed to solve the problem.
- Your task is to explain in detail how the code snippet works to solve the problem.
- Provide a step-by-step explanation of the logic and approach used in the code snippet.
</task>

<problem>
{question_content}
</problem>

<code_snippet>
{code_snippet}
</code_snippet>
"""

In [317]:
C0_explanation = llm_client.generate(
    EXPLAIN_PROMPT_TEMPLATE.format(
        question_content=problem.question_content,
        code_snippet=C0_snippet
)
)

C1_explanation = llm_client.generate(
    EXPLAIN_PROMPT_TEMPLATE.format(
        question_content=problem.question_content,
        code_snippet=C1_snippet
)
)

print("C0 Explanation:")
print(C0_explanation)
print("\nC1 Explanation:")
print(C1_explanation)

2025-12-10 15:31:24.477 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal
2025-12-10 15:31:57.949 | DEBUG    | common.llm_client:generate:302 - Generated 7006 characters
2025-12-10 15:31:57.950 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal
2025-12-10 15:32:30.464 | DEBUG    | common.llm_client:generate:302 - Generated 7000 characters


C0 Explanation:
I'll explain how this code works to solve the problem, step by step.

High-level idea
- The array nums is [1..n]. For any starting index i, if the smallest element j > i that conflicts with some element in the prefix starting at i (i.e., there exists a conflicting pair [x,y] with x in [i..j-1] and y = j or vice-versa) is at position R(i), then every subarray that starts at i and ends at a position < R(i) is valid (it does not contain both elements of any remaining conflicting pair). The number of valid subarrays starting at i is therefore max(0, R(i)-i).
- The code builds, for each start index i, a "blocked_right" array where blocked_right[i] = R(i) (or INF if there is no blocking element). The total number of valid subarrays given a set of conflicting pairs is the sum over i of max(0, blocked_right[i] - i).
- We must remove exactly one conflicting pair and return the maximum total valid subarrays obtainable. Removing a pair [a,b] only affects blocking information for s

# DET Prompt

In [217]:
DET_PROMPT = """
<task>
- You are given a programming problem described in a <problem> block. 
- You are also given two Python code snippets in <solution variant='P'> and <solution variant='Q'> blocks that are intended to solve the same programming problem. 
- Additionally, you have access to explanations of how each code snippet works in the <explanation variant='P'> and <explanation variant='Q'> blocks.
- You are also given the current test inputs and the corresponding equivalent outputs they produce for both P and Q solutions in a <current_tests> block.
- The current tests fail to differentiate between the two code snippets as they produce the same outputs for both.
- Your task is to generate a new test input that can differentiate between the two code snippets.
- Consider the following guidelines while generating the test input:
    -- Analyze the problem statement to understand the requirements and constraints.
    -- In the problem, identify the input format, and input constraints, and any special conditions that need to be handled.
    -- Review the code solutions with the explanations provided to identify any differences in logic, approach, or edge case handling between the two.
    -- Think of edge cases, boundary conditions, very large input cases, and scenarios that might lead to different behaviors in the two implementations.
    -- Consider inputs that test the limits of the problem constraints or that exploit any identified weaknesses in either code snippet.
    -- New test input should be completely different from the existing test inputs provided in the <current_tests> block.
</task>

<problem>
{question_content}
</problem>


<solution variant='P'>
```python
{code_snippet_P}
```
</solution>

<explanation variant='P'>
{explanation_P}
</explanation>

<solution variant='Q'>
```python
{code_snippet_Q}
```
</solution>

<explanation variant='Q'>
{explanation_Q}
</explanation>

<current_tests>
{current_tests}
</current_tests>

<output_format>
Provide a single test input in the following Python dict format inside a code block:
```python
{{"inputdata": {{<your_generated_test_input>"}}}}
```

eg: for stdin test case:
```python
{{"inputdata": {{"input_str": "test input string"}}}}
```
for functional test case:
```python
{{"inputdata": {{"param1": value1, "param2": value2, ...}}}}
```

</output_format>
"""

In [218]:
import random
current_tests = public_test_input_outputs.copy()

for i in range(10):
    print(f"Generating differential test input, attempt {i+1}/10...")
    prompt = DET_PROMPT.format(
        question_content=problem.question_content,
        code_snippet_P=C0_snippet,
        explanation_P=C0_explanation,
        code_snippet_Q=C1_snippet,
        explanation_Q=C1_explanation,
        current_tests="\n".join(json.dumps(t) for t in random.sample(current_tests, min(len(current_tests), 20))),
    )
    response = llm_client.generate(prompt=prompt)

    test_input = extract_code_block_from_response(response)
    print("Generated differential test input:", test_input)

    c0_script = compose_lcb_output_script(c0_ind.snippet, test_input)
    c0_result = sandbox.execute_code(c0_script).output.strip()

    c1_script = compose_lcb_output_script(c1_ind.snippet, test_input)
    c1_result = sandbox.execute_code(c1_script).output.strip()


    if c0_result != c1_result:
        print(f"Differential test input found on attempt {i+1}!")
        print("Test Input:", test_input)
        print("C0 Output:", c0_result)
        print("C1 Output:", c1_result)
        break

    else:
        print(f"No discrepancy found on attempt {i+1}. Trying again...")
        current_tests.append({'input': json.loads(test_input)["inputdata"], 'output': c0_result})



2025-12-09 13:16:34.830 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal


Generating differential test input, attempt 1/10...


2025-12-09 13:16:38.737 | DEBUG    | common.llm_client:generate:302 - Generated 84 characters
2025-12-09 13:16:38.743 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpk8zjn2rm.py capture_output=True code_len=2447
2025-12-09 13:16:38.788 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:16:38.791 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmp5zl421vh.py capture_output=True code_len=3361
2025-12-09 13:16:38.820 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:16:38.821 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal


Generated differential test input: {"inputdata": {"n": 6, "conflictingPairs": [[1,6],[1,5],[2,4],[2,6]]}}
No discrepancy found on attempt 1. Trying again...
Generating differential test input, attempt 2/10...


2025-12-09 13:16:41.594 | DEBUG    | common.llm_client:generate:302 - Generated 90 characters
2025-12-09 13:16:41.600 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpn0t5g4v1.py capture_output=True code_len=2455
2025-12-09 13:16:41.646 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:16:41.649 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmprz2kfeif.py capture_output=True code_len=3369
2025-12-09 13:16:41.680 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:16:41.681 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal


...

Generated differential test input: {"inputdata": {"n": 8, "conflictingPairs": [[2,8],[2,7],[3,8],[4,6],[1,8],[1,7],[5,8]]}}
No discrepancy found on attempt 8. Trying again...
Generating differential test input, attempt 9/10...


2025-12-09 13:16:58.525 | DEBUG    | common.llm_client:generate:302 - Generated 90 characters
2025-12-09 13:16:58.531 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpqytj8p25.py capture_output=True code_len=2455
2025-12-09 13:16:58.573 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:16:58.575 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpwdz_kmxm.py capture_output=True code_len=3369
2025-12-09 13:16:58.604 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:16:58.605 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal


Generated differential test input: {"inputdata": {"n": 6, "conflictingPairs": [[1,6],[1,5],[2,6],[2,5],[3,4]]}}
No discrepancy found on attempt 9. Trying again...
Generating differential test input, attempt 10/10...


2025-12-09 13:17:01.613 | DEBUG    | common.llm_client:generate:302 - Generated 90 characters
2025-12-09 13:17:01.618 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpsh9cebzl.py capture_output=True code_len=2455
2025-12-09 13:17:01.662 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:17:01.664 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmp2z6v7ohc.py capture_output=True code_len=3369
2025-12-09 13:17:01.706 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0


Generated differential test input: {"inputdata": {"n": 6, "conflictingPairs": [[1,6],[2,6],[3,6],[1,5],[2,5]]}}
No discrepancy found on attempt 10. Trying again...


# Input Generation

In [282]:
INPUT_GENERATION_PROMPT = """
<task>
- You are given a programming problem described in a <problem> block. 
- You are also given two Python code snippets in <solution variant='P'> and <solution variant='Q'> blocks that are intended to solve the same programming problem. 
- Additionally, you have access to explanations of how each code snippet works in the <explanation variant='P'> and <explanation variant='Q'> blocks.
- You are also given the current test inputs and the corresponding equivalent outputs they produce for both P and Q solutions in a <current_tests> block.
- The current tests fail to differentiate between the two code snippets as they produce the same outputs for both.
- Your task is to write a python script that generates test inputs that can differentiate between the two code snippets.
- The script should define a function named `generate_test_inputs` that takes an integer parameter specifying the number of test inputs to generate and returns a list of dictionaries, each representing a test input.
- The script should be able to generate multiple diverse test inputs as much as required by the user in a single execution. 
    -- Eg: if the user requests 100 test inputs, the script should generate 100 test inputs in one go.
- You are allowed to use standard python libraries to implement the script.
- You are allowed to define helper functions within the script to aid in generating the test inputs.
- You are NOT allowed to manually hardcode any test inputs within the script.
</task>

<problem>
{question_content}
</problem>

<solution variant='P'>
```python
{code_snippet_P}
```
</solution>

<solution variant='Q'>
```python
{code_snippet_Q}
```
</solution>


<current_tests>
{current_tests}
</current_tests>

<output_format>
Provide only the python script inside a code block as follows:
```python
def generate_test_inputs(num_inputs: int) -> list[dict]:
    # Your code to generate test inputs
```
if input is via stdin, each dict should be of the form:
{{"input_str": "<generated_input_string>"}}
if input is functional, each dict should be of the form:
{{"param1": value1, "param2": value2, ...}}
</output_format>

"""

In [283]:
from common.code_preprocessing.transformation import remove_if_main_block
response = llm_client.generate(
    INPUT_GENERATION_PROMPT.format(
        question_content=problem.question_content,
        code_snippet_P=C0_snippet,
        code_snippet_Q=C1_snippet,
        current_tests="\n".join(json.dumps(t) for t in public_test_input_outputs),
)
)

generated_script = extract_code_block_from_response(response)
generated_script = remove_if_main_block(generated_script)
generated_script += """\n
print(generate_test_inputs(100))
"""

2025-12-09 13:49:24.589 | DEBUG    | common.llm_client:generate:282 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal
2025-12-09 13:49:50.300 | DEBUG    | common.llm_client:generate:302 - Generated 4740 characters


In [298]:
print(generated_script)

def generate_test_inputs(num_inputs: int) -> list[dict]:
    """
    Generate test cases for the 'maxSubarrays' problem.
    Each test is returned as a dict with keys 'n' and 'conflictingPairs' (functional format).
    The generator produces diverse cases (small, medium, edge, and random) without hardcoding specific instances.
    """
    import random
    from math import ceil

    def make_random_case(n, m):
        pairs = set()
        while len(pairs) < m:
            a = random.randint(1, n)
            b = random.randint(1, n)
            if a == b:
                continue
            x, y = (a, b) if a < b else (b, a)
            pairs.add((x, y))
        return {'n': n, 'conflictingPairs': [list(p) for p in pairs]}
    tests = []
    random.seed(12648430)
    generators = []
    generators.append(lambda: {'n': 2, 'conflictingPairs': [[1, 2]]})
    generators.append(lambda: {'n': 3, 'conflictingPairs': [[1, 2], [1, 3], [2, 3]]})

    def shared_a_case():
        n = 10
       

In [299]:
print(sandbox.execute_code(generated_script))

2025-12-09 13:53:41.302 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpno7ed6b8.py capture_output=True code_len=3438
2025-12-09 13:53:41.355 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0


BasicExecutionResult(success=True, output="[{'n': 2, 'conflictingPairs': [[1, 2]]}, {'n': 3, 'conflictingPairs': [[1, 2], [1, 3], [2, 3]]}, {'n': 10, 'conflictingPairs': [[1, 4], [2, 4], [4, 5], [4, 7], [4, 8]]}, {'n': 12, 'conflictingPairs': [[1, 12], [2, 11], [3, 10], [4, 9], [5, 8], [6, 7], [11, 12], [6, 11]]}, {'n': 1000, 'conflictingPairs': [[166, 782], [434, 936], [852, 956]]}, {'n': 2000, 'conflictingPairs': [[132, 268], [1819, 1987], [20, 621], [14, 1215], [672, 1072], [432, 974], [372, 780], [1032, 1235], [498, 1822], [540, 1001], [112, 1643], [407, 773], [232, 1824], [687, 827], [822, 1775], [1628, 1759], [492, 588], [188, 1000], [91, 393], [71, 186], [700, 1669], [591, 1762], [162, 1356], [706, 1191], [1695, 1785], [1039, 1251], [113, 268], [1104, 1609], [711, 1943], [1485, 1566], [61, 486], [1288, 1546], [693, 1108], [165, 1626], [752, 1343], [36, 1890], [934, 1957], [303, 693], [328, 606], [875, 1907], [250, 1706], [584, 1960], [148, 675], [1003, 1189], [524, 1420], [1118,

In [300]:
from common.code_preprocessing.analysis import validate_code_syntax
import ast

output = sandbox.execute_code(generated_script).output
if not validate_code_syntax(output):
    raise ValueError("Generated script has syntax errors.")


test_inputs: list[dict] = ast.literal_eval(output)
print("Number of generated test inputs:", len(test_inputs))
print(test_inputs)

2025-12-09 13:53:43.334 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpsoy97d61.py capture_output=True code_len=3438
2025-12-09 13:53:43.400 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0


Number of generated test inputs: 100
[{'n': 2, 'conflictingPairs': [[1, 2]]}, {'n': 3, 'conflictingPairs': [[1, 2], [1, 3], [2, 3]]}, {'n': 10, 'conflictingPairs': [[1, 4], [2, 4], [4, 5], [4, 7], [4, 8]]}, {'n': 12, 'conflictingPairs': [[1, 12], [2, 11], [3, 10], [4, 9], [5, 8], [6, 7], [11, 12], [6, 11]]}, {'n': 1000, 'conflictingPairs': [[166, 782], [434, 936], [852, 956]]}, {'n': 2000, 'conflictingPairs': [[132, 268], [1819, 1987], [20, 621], [14, 1215], [672, 1072], [432, 974], [372, 780], [1032, 1235], [498, 1822], [540, 1001], [112, 1643], [407, 773], [232, 1824], [687, 827], [822, 1775], [1628, 1759], [492, 588], [188, 1000], [91, 393], [71, 186], [700, 1669], [591, 1762], [162, 1356], [706, 1191], [1695, 1785], [1039, 1251], [113, 268], [1104, 1609], [711, 1943], [1485, 1566], [61, 486], [1288, 1546], [693, 1108], [165, 1626], [752, 1343], [36, 1890], [934, 1957], [303, 693], [328, 606], [875, 1907], [250, 1706], [584, 1960], [148, 675], [1003, 1189], [524, 1420], [1118, 1521]

In [301]:
for idx, ti in enumerate(test_inputs):
    ti_formatted = {"inputdata": ti}
    c0_script = compose_lcb_output_script(c0_ind.snippet, str(ti_formatted))
    c0_result = sandbox.execute_code(c0_script).output.strip()
    c1_script = compose_lcb_output_script(c1_ind.snippet, str(ti_formatted))
    c1_result = sandbox.execute_code(c1_script).output.strip()
    if c0_result != c1_result:
        print(f"Test Input {idx+1}:\n")
        print("Discrepancy found!")
        print("Test Input:", ti)
        print("C0 Output:", c0_result)
        print("C1 Output:", c1_result)
        break

2025-12-09 13:53:49.639 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpr_lj45w2.py capture_output=True code_len=2423
2025-12-09 13:53:49.676 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:53:49.679 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmp4xbcnlvi.py capture_output=True code_len=3337
2025-12-09 13:53:49.708 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:53:49.710 | DEBUG    | common.sandbox:execute_code:543 - Executing code in sandbox: temp_file=/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpnga76f76.py capture_output=True code_len=2439
2025-12-09 13:53:49.737 | DEBUG    | common.sandbox:execute_code:559 - Execution finished: returncode=0
2025-12-09 13:53:49.739 | DEBUG    | common.sandbox:execute_code:543 -